# 00_02 Reading error messages

*Our own page: not in the book.*

You already know how to read a traceback in your own language, so we'll skip the basics. This page is only about the errors that come from this course: notebooks, PyTorch and our own files. We really did hit every one of them while building the project.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **getting ready**: nothing to build yet
>
> **This notebook:** Sooner or later a cell stops with a red error message. Here are the notebook and PyTorch errors you're most likely to hit, and what each one means.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- The notebook error you'll meet first: a name that 'is not defined'
- The three PyTorch errors behind most beginner problems: shape, data type and device
- A real traceback through our own `tokenizer.py`

## 2. The idea

### ❓ Why do I need a page on errors if I can already read a traceback?

Because in ML code the error often points at the **wrong place**. A shape mistake in a model gets reported on the line that *uses* the bad shape, not the line that *made* it. If you know the three or four usual suspects, you save yourself hours.

Same rule as always, though: read the last line first, then look above it for a line in your own file.

## 3. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ I ran a cell and got `NameError: name '…' is not defined`, but the name is right there in an earlier cell. Why?

Because that earlier cell **hasn't run yet**. The notebook's kernel (the Python process behind it) only knows the names made by cells you've actually run, not the ones you can see on the page. It happens when you run a cell before the ones above it, or after a kernel restart, which wipes its memory.

The fix: run the cells above it, in order, or click Run All. That's why the course rule is to run from the top. This one causes most of the errors you'll see.

👉 **Predict first:** what will the last line of the traceback say?

In [ ]:
import traceback

try:
    print(epochs)  # nobody ever created `epochs`
except NameError:
    traceback.print_exc()  # print the error, but let the notebook carry on

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The last line is `NameError: name 'epochs' is not defined`. (The cells on this page catch their own errors, which is why Run All still works here.)

</details>

### ❓ ModuleNotFoundError: what does it mean here?

In this course it nearly always means the notebook is on the **wrong kernel**, a Python that doesn't have our libraries (torch, tiktoken…). The fix: click **Select Kernel** and pick the Python in `.venv`.

### ❓ Shape mismatch: what does it look like?

A **tensor** is PyTorch's grid of numbers, and its **shape** is its size in each direction: `(2, 3)` means 2 rows and 3 columns (more in [A_02](../A_toolbox/A_02_pytorch_tensors.ipynb)). Matrix multiplication (`@`) needs the two inside numbers to match: `(2, 3) @ (3, 4)` works, `(2, 3) @ (2, 3)` doesn't. It's the most common PyTorch error, and at least the message prints both shapes.

👉 **Predict first:** which two shapes will the message show?

In [ ]:
import torch

try:
    torch.ones(2, 3) @ torch.ones(2, 3)
except RuntimeError:
    traceback.print_exc()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

So: `shapes cannot be multiplied (2x3 and 2x3)`. The thing is, the fix is almost never on this line. You have to find where the shape went wrong earlier, printing `.shape` along the way.

</details>

### ❓ Data type mismatch: what does it look like?

Every tensor has a **dtype** (data type). It holds either whole numbers (`int64`, which PyTorch calls **Long**) or decimals (`float32`, called **Float**). Token IDs, the numbers the tokenizer gives each character, are whole numbers. Weights and scores are decimals. Mix them up and you get an error.

👉 **Predict first:** an **embedding table** is a lookup table with one row of numbers per token ID (chapter 2 builds one). What happens if we give it decimals instead of IDs?

In [ ]:
try:
    torch.nn.Embedding(5, 2)(torch.tensor([1.0, 2.0]))  # IDs must be whole numbers
except RuntimeError:
    traceback.print_exc()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

There it is: `Expected tensor … Long, Int; but got … FloatTensor`. The fix is to build the IDs as whole numbers (`torch.tensor([1, 2])`).

</details>

### ❓ Device mismatch: what is that?

A **device** is where a tensor lives: the normal CPU, or a GPU (`cuda`, or `mps` on a Mac). If one tensor is on the GPU and the other on the CPU, you get *Expected all tensors to be on the same device*. Everything in this course stays on the CPU, so you shouldn't meet it here. You will the day you move to a GPU, though.

### ❓ What if a change makes no error but the results are wrong?

That's the worst kind. Once, in our `train()` function, the lines that update the model slipped one level out of the inner loop, so the model learned from only one chunk of text per **epoch** (one pass through the training text). Nothing crashed. We only caught it because the output stopped matching our saved results. So the lesson is: **compare results before and after a change** ([05_04](../05_training_on_the_odyssey/05_04_running_fair_experiments.ipynb) shows how).

### ❓ I see 'Failed to initialize NumPy'. Is something broken?

No, it's only a **warning**. A warning means "this isn't ideal, but I'm carrying on", while an error stops the program. PyTorch is saying it can't find NumPy, another maths library. It's harmless. NumPy gets installed with the project, so running `uv sync` again makes it go away.

## 4. Full example

### ❓ Can I read a real traceback from deep inside our code?

Let's find out. Here we ask our own tokenizer to decode an ID that doesn't exist (the vocabulary only goes up to 136).

👉 **Predict first:** what kind of error, and which of our files will the traceback point to?

In [ ]:
from homer_ai.tokenizer import CharTokenizer
from homer_ai.train_homer import BOOKS_DIR, load_books

tokenizer = CharTokenizer("\n".join(load_books(BOOKS_DIR)))

try:
    tokenizer.decode([41, 9999])
except KeyError:
    traceback.print_exc()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

This time the last line is `KeyError: 9999`. Just above it, the traceback points into **our** file, `tokenizer.py`, inside `decode`. That's where to look: the dictionary `id_to_char` simply has no entry 9999.

</details>

## 5. Recap

- Read the last line first, then find your own file's line. In a notebook, a NameError usually means a cell above hasn't run.
- Shape, dtype and device are the three PyTorch mismatches. Print `.shape` and `.dtype` to track them down.
- A warning isn't an error: the program carries on.

**Next:** Chapter 1: [Foundations](../01_foundations/01_00_chapter_overview.ipynb).